# RadioSkySims — Mapas do Céu

Fluxo de exemplo do bounded context **SkyMap** (`radioskysims-skymap`):
geração de mapas HEALPix (sintético offline + PySM3), operações
(suavização/máscara), análise (estatística + espectro angular Cℓ) e I/O (FITS).

A visualização é Mollweide via HEALPy, com figuras salvas em `plots/`.


## API unificada — `SkyMapOrchestrator`

Para uniformizar a experiência do usuário, o pacote `radioskysims-skymap`
expõe `SkyMapOrchestrator` como fachada pública de alto nível para notebooks
e scripts exploratórios:

```python
from radioskysims_skymap import SkyMapOrchestrator

sky = SkyMapOrchestrator().from_coefficients(cls, nside=64, seed=42)
sky = SkyMapOrchestrator().load("mapa.fits")

noise = SkyMapOrchestrator().from_noise(
    method="pysm",
    nside=64,
    freq_ghz=1.0,
)

sky.smooth(fwhm_deg=5.0).rotate(
    rot_deg=(0, 0, 0)
).apply(mask).view()

sky.save("mapa.fits", format="fits")
sky.power_spectrum()
sky.statistics()
sky.register("meu-mapa")
```

As seções abaixo demonstram geração, operações, análise e persistência
utilizando a fachada pública `SkyMapOrchestrator`.


## 1. Parâmetros de referência


In [ ]:
# ── Âncora de diretório: raiz do projeto (CWD-independente) ──
# Garante que caminhos relativos (ex.: Path("data")) resolvam da raiz do
# projeto, independente de onde o kernel Jupyter iniciou (notebooks/ ou raiz).
import os

from radioskysims_core.paths import get_project_root

os.chdir(get_project_root())
print("cwd →", os.getcwd())

In [ ]:
# ── Parâmetros de referência para mapas do céu ──
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from radioskysims_viz import save_figure

# Resolução HEALPix (potência de 2) e frequência de referência.
NSIDE = 64  # → 49152 pixels
FREQ_GHZ = 1.0  # 1 GHz (banda L) — usado nos metadados/PySM
FRAME = "galactic"  # galactic | equatorial | ecliptic
SEED = 42

DATA_DIR = Path("data")
SKYMAP_DIR = DATA_DIR / "skymap"
SKYMAP_DIR.mkdir(parents=True, exist_ok=True)
print(f"NSide={NSIDE} → {hp.nside2npix(NSIDE)} pixels | freq={FREQ_GHZ} GHz | frame={FRAME}")


## 2. Mapa sintético (offline) — geração e visualização

Campo aleatório com espectro tipo-físico (`Cℓ ∝ ℓ^{-3/2}`) via `hp.synfast` — determinístico (seed) e sem rede.


In [ ]:
# Mapa sintético OFFLINE via SkyMapOrchestrator.
from radioskysims_skymap import SkyMapOrchestrator

rng = np.random.default_rng(SEED)
ell = np.arange(3, 200)
cls_input = 1.0 * ell**-1.5  # ~ Cℓ ∝ ℓ^{-3/2}

analytic = SkyMapOrchestrator().from_coefficients(
    cls_input,
    nside=NSIDE,
    lmax=199,
    seed=SEED,
    unit="uK_CMB",
    frame=FRAME,
    freq_ghz=FREQ_GHZ,
)

analytic
analytic.view(
    show=False,
    title="Mapa sintético (Cℓ) — SkyMapOrchestrator",
)
save_figure(plt.gcf(), "skymap_analytic")


## 3. Geração com PySM3 (componentes físicos)

PySM3 monta mapas por componentes (synchrotron, dust, free-free, AME, CMB).
**Requer download de templates na 1ª execução** — se indisponível, apenas avisa.


In [ ]:
# PySM3: componentes físicos via SkyMapOrchestrator (opcional).
try:
    noise = SkyMapOrchestrator().from_noise(
        nside=NSIDE,
        freq_ghz=FREQ_GHZ,
        method="pysm",
        components=["synchrotron", "dust"],
        unit="uK_CMB",
        frame=FRAME,
        seed=SEED,
    )

    noise
    noise.view(
        show=False,
        title="PySM3: synchrotron + dust — SkyMapOrchestrator",
    )
    save_figure(plt.gcf(), "skymap_pysm")
except Exception as exc:
    print(
        "[aviso] PySM3 indisponível (templates?) —",
        type(exc).__name__,
        str(exc)[:200],
    )


## 4. Operações: suavização e máscara

`SkyMapOperations.smooth` (gaussiana) e `SkyMask.apply` (máscara binária) — domínio puro, sem I/O.


In [ ]:
# Suavização gaussiana (FWHM em graus) via SkyMapOrchestrator.
FWHM_DEG = 5.0

smoothed = SkyMapOrchestrator.from_handle(
    analytic.handle
).smooth(fwhm_deg=FWHM_DEG)

smoothed
smoothed.view(
    show=False,
    title=f"Suavizado FWHM={FWHM_DEG}° — SkyMapOrchestrator",
)
save_figure(plt.gcf(), "skymap_smoothed")


In [ ]:
# Máscara binária (metade do céu) via SkyMapOrchestrator.apply().
mask_half = np.ones(analytic.data.size)
mask_half[analytic.data.size // 2 :] = 0.0

masked = SkyMapOrchestrator.from_handle(
    analytic.handle
).apply(mask_half)

masked
masked.view(
    show=False,
    title="Máscara (metade do céu) — SkyMapOrchestrator",
)
save_figure(plt.gcf(), "skymap_masked")


## 5. Análise: estatística e espectro angular

`SkyMapAnalysis` (média/RMS/min-max/Cℓ) + `SkyMapVisualizer.plot_cls`.


In [ ]:
# Estatísticas + espectro angular Cℓ via SkyMapOrchestrator.
stats = analytic.statistics()

print("mean:", round(stats.mean, 3), "uK_CMB")
print("rms :", round(stats.rms, 3), "uK_CMB")
print("min :", round(stats.min, 3), "| max:", round(stats.max, 3), "uK_CMB")

cls = analytic.power_spectrum(lmax=200)
print("Cℓ (ℓ=0..4):", [round(float(c), 3) for c in cls[:5]])

# Plot do espectro Cℓ.
ell_all = np.arange(len(cls))
plt.figure()
plt.loglog(ell_all[2:], cls[2:])
plt.xlabel("ℓ")
plt.ylabel("Cℓ")
plt.title("Espectro angular (Cℓ) — SkyMapOrchestrator")
save_figure(plt.gcf(), "skymap_cls")


## 6. I/O: FITS

`HealpySkyMapProvider` grava e relê mapas em FITS.


In [ ]:
# I/O FITS via SkyMapOrchestrator.
path = SKYMAP_DIR / "skymap_analytic.fits"

analytic.save(
    path,
    format="fits",
    overwrite=True,
)
print("salvo:", path)

reloaded = SkyMapOrchestrator().load(
    path,
    unit="uK_CMB",
    frame=FRAME,
)

reloaded
print(
    "igual ao original:",
    np.allclose(reloaded.data, analytic.data),
)

## 7. API unificada — `SkyMapOrchestrator` (recomendado)

Mesmo fluxo das seções anteriores — geração, suavização, máscara,
análise e persistência — agora encadeado utilizando somente a fachada
pública `SkyMapOrchestrator`.


In [ ]:
from radioskysims_skymap import SkyMapOrchestrator

# Geração usando o espectro Cℓ definido anteriormente.
result = SkyMapOrchestrator().from_coefficients(
    cls_input,
    nside=NSIDE,
    lmax=199,
    seed=SEED,
    unit="uK_CMB",
    freq_ghz=FREQ_GHZ,
)

mask_half_sky = np.ones(result.data.size)
mask_half_sky[len(mask_half_sky) // 2 :] = 0.0

# Operações encadeáveis na fachada pública.
result.smooth(
    fwhm_deg=FWHM_DEG,
).apply(
    mask_half_sky,
).view(
    show=False,
    title="SkyMapOrchestrator: smooth + mask",
)

fig = plt.gcf()
fig
save_figure(fig, "skymap_orchestrator_chain")

# Análise.
stats = result.statistics()
print(
    f"mean={stats.mean:.3f} "
    f"rms={stats.rms:.3f} "
    f"min={stats.min:.3f} "
    f"max={stats.max:.3f} uK_CMB"
)

cls_out = result.power_spectrum(lmax=200)
print(
    "Cℓ (ℓ=0..4):",
    [round(float(c), 3) for c in cls_out[:5]],
)

# Persistência: FITS + registro HDF5 por chave lógica.
orchestrator_path = SKYMAP_DIR / "skymap_orchestrator.fits"

result.save(
    orchestrator_path,
    format="fits",
    overwrite=True,
)

result.register(
    "skymap-orchestrator-demo",
    overwrite=True,
)

reloaded = SkyMapOrchestrator().load_registered(
    "skymap-orchestrator-demo"
)

print(
    "registro íntegro:",
    np.allclose(reloaded.data, result.data),
)
